# C12-classical-models — Session 4: Decision Trees from Impurity to Prediction

*One 90-minute session. Binary labels use integers `0` and `1`; features are float arrays.*


In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

SEED = 20260804
ATOL = 1e-10
RTOL = 1e-8
rng = np.random.default_rng(SEED)


## 1. Node distributions, Gini impurity, and entropy

At a node with class proportions $p_1,\dots,p_K$, Gini impurity is
$G=1-\sum_kp_k^2$ and entropy is $H=-\sum_{k:p_k>0}p_k\log_2p_k$. Both are zero for a pure node
and largest at a balanced distribution. A leaf predicts the most frequent class; ties choose the
smaller class label in this unit.

For binary counts `(3,1)`, proportions are `(3/4,1/4)` and Gini is
$1-9/16-1/16=3/8$.

**Checkpoint 1A.** Compute Gini for binary counts `(2,2)`.

**Checkpoint 1B.** Why are zero-probability entropy terms omitted?


In [ ]:
def gini(y):
    y = np.asarray(y, dtype=np.int64)
    if y.size == 0:
        return 0.0
    counts = np.bincount(y)
    p = counts[counts > 0] / y.size
    return float(1.0 - p @ p)

assert np.isclose(gini(np.array([0, 0, 1, 1])), 0.5, atol=ATOL, rtol=RTOL)
assert np.isclose(gini(np.array([0, 0, 0, 1])), 0.375, atol=ATOL, rtol=RTOL)


## 2. Candidate thresholds and weighted child impurity

For feature $j$, sort its distinct observed values. Candidate thresholds are midpoints between
adjacent distinct values. A row goes left when $x_j\le\theta$ and right otherwise. Empty children
are invalid. For parent size $n$, weighted child impurity is

$$I_{children}=\frac{n_L}{n}I_L+\frac{n_R}{n}I_R,$$

and information gain is $I_{parent}-I_{children}$. Maximizing gain equals minimizing weighted
child impurity because the parent is fixed.

**Checkpoint 2A.** Why not use an observed value itself as the only threshold convention?

**Checkpoint 2B.** What is gain when a split leaves child impurity equal to parent impurity?


## 3. Deterministic best-split search

This unit searches features in increasing index order and thresholds in increasing numeric order.
It selects the smallest tuple `(weighted_impurity, feature_index, threshold)` lexicographically;
thus exact impurity ties go to the smaller feature index, then smaller threshold. Comparisons use
the computed float values; tests pin data whose intended ties are exact.

The function returns `(feature, threshold, weighted_impurity)` or `None` if no valid split exists.

**Checkpoint 3A.** If two exact ties use features 0 and 2, which wins?

**Checkpoint 3B.** Within feature 0, which of tied thresholds 1.5 and 3.5 wins?


In [ ]:
def best_split(X, y):
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=np.int64)
    best = None
    for feature in range(X.shape[1]):
        values = np.unique(X[:, feature])
        thresholds = (values[:-1] + values[1:]) / 2.0
        for threshold in thresholds:
            left = X[:, feature] <= threshold
            if not left.any() or left.all():
                continue
            weighted = (left.mean() * gini(y[left]) +
                        (~left).mean() * gini(y[~left]))
            candidate = (weighted, feature, float(threshold))
            if best is None or candidate < best:
                best = candidate
    return None if best is None else (best[1], best[2], best[0])

X_split = np.array([[0., 0.], [1., 1.], [2., 0.], [3., 1.]])
y_split = np.array([0, 0, 1, 1])
split = best_split(X_split, y_split)
assert split[0] == 0
assert np.isclose(split[1], 1.5, atol=ATOL, rtol=RTOL)
assert np.isclose(split[2], 0.0, atol=ATOL, rtol=RTOL)
print(split)


## 4. Recursion, leaves, and prediction

To grow a classification tree: make a leaf when the node is pure, reaches `max_depth`, has fewer
than `min_samples_split` rows, or has no split with positive gain. Otherwise store the best
`feature` and `threshold`, then recurse left and right. Depth 0 is the root. Each leaf stores its
majority class using the smaller-label tie rule.

Prediction starts at the root and follows `<= threshold` left until a leaf. Training must carry
row subsets, while prediction needs only stored node fields. A depth-limited implementation can
use nested dictionaries with keys `feature`, `threshold`, `left`, `right`, or `prediction`.

**Checkpoint 4A.** Can a node at `depth == max_depth` split again?

**Checkpoint 4B.** What must happen if the best gain is zero?


## 5. Worked tree construction

For one feature values `(0,1,2,3)` and labels `(0,0,1,1)`, candidate thresholds are
$0.5,1.5,2.5$. Threshold $1.5$ produces pure children, weighted Gini zero, and parent gain
$0.5$. The root stores feature 0 and threshold 1.5; its left leaf predicts 0 and right leaf 1.
The decision regions are axis-aligned intervals.

**Checkpoint 5A.** Predict inputs 1.5 and 1.5001 under the `<=` convention.

**Checkpoint 5B.** What is the depth of both leaves in this stump?


In [ ]:
tree = DecisionTreeClassifier(criterion="gini", max_depth=1,
                              random_state=SEED)
tree.fit(X_split[:, [0]], y_split)
pred = tree.predict(np.array([[1.5], [1.5001]]))
assert np.array_equal(pred, np.array([0, 1]))
print("threshold", tree.tree_.threshold[0], "| predictions", pred)


## 6. Stopping, regularization, and pruning

Unrestricted trees can isolate noise: low training error but high validation variance.
Pre-pruning controls include `max_depth`, `min_samples_split`, `min_samples_leaf`, and
`max_features`. Cost-complexity post-pruning chooses a subtree by trading empirical leaf impurity
against number of leaves: $R_\alpha(T)=R(T)+\alpha|T_{leaves}|$. In scikit-learn this is
controlled by `ccp_alpha`; larger values favor smaller trees.

Choose controls by cross-validation on training data. Report depth, leaf count, train score, and
validation score rather than claiming “shallower is always better.”

**Checkpoint 6A.** Which parameter directly enforces a minimum number of rows per leaf?

**Checkpoint 6B.** What model-size quantity does cost-complexity pruning penalize here?


## 7. Comparison axes, pitfalls, and exam connections

Trees are supervised, minimize impurity greedily rather than a smooth global loss, form
axis-aligned nonlinear regions, and are largely insensitive to monotone feature scaling. Small
trees are directly interpretable as rules; deep trees have high variance. `predict_proba` returns
leaf class frequencies, which are not automatically well calibrated. Validate depth/pruning with
classification metrics and cross-validation.

Compared with logistic/SVM models, trees need no coefficient geometry or gradient step, but split
ties and stopping rules become part of the algorithmic contract.

**Pitfalls.** Using unweighted child impurity favors tiny children; maximizing impurity reverses
the split; inconsistent `<` versus `<=` changes boundary rows; failing to stop zero-gain recursion
can loop; and arbitrary set iteration destroys deterministic ties.

**Exam connection.** Expect a hand-computed impurity table or a constrained splitter with exact
ties. **Going deeper.** Session 5 averages or sequentially corrects trees.

**Checkpoint 7A.** Why usually scale an SVM but not a decision tree?

**Checkpoint 7B.** Name the deterministic tie hierarchy for split search.


## Checkpoint answers

**1A.** $0.5$. **1B.** $0\log 0$ is defined by its limit as zero; omitting it avoids an undefined
numeric logarithm.

**2A.** Midpoints make the partition explicit between observed values and avoid duplicate
equivalent cuts. **2B.** Zero.

**3A.** Feature 0. **3B.** Threshold 1.5.

**4A.** No. **4B.** Emit a majority-class leaf.

**5A.** 0 and 1. **5B.** Depth 1.

**6A.** `min_samples_leaf`. **6B.** Number of leaves.

**7A.** SVM Euclidean norms/distances change with scale; tree order comparisons do not.
**7B.** Minimum weighted impurity, then smaller feature index, then smaller threshold.
